<a href="https://colab.research.google.com/github/SaimNaveed646/ML_Models/blob/main/RandomForest/RandomForest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ensemble Learning

Ensemble learning is a machine learning technique where multiple models are combined to make one final prediction. The idea is that a group of models usually performs better than a single model.

Imagine asking one doctor for a diagnosis vs asking 10 doctors and taking the majority opinion.The second approach is usually more reliable. Ensemble learning works in the same way.

# How predictions are combined
## For Classification
Hard voting and soft voting are two ways to combine predictions in ensemble learning when you have multiple classifiers.
Suppose we have 3 models:
| Model   | Prediction |
| ------- | ---------- |
| Model 1 | Cat        |
| Model 2 | Dog        |
| Model 3 | Dog        |
final prediction -> Dog (because it has the majority vote)
So the ensemble simply selects the most common class.


In soft voting, each model predicts class probabilities instead of just labels.
The probabilities are averaged, and the class with the highest average probability is selected.

| Model   | P(Cat) | P(Dog) |
| ------- | ------ | ------ |
| Model 1 | 0.30   | 0.70   |
| Model 2 | 0.30   | 0.70   |
| Model 3 | 0.80   | 0.20   |

* P(Cat) = (0.30 + 0.30 + 0.80) / 3 = 0.46
* P(Dog) = (0.70 + 0.70 + 0.20) / 3 = 0.53

final prediction -> Dog

## For Regression -> Use average
* Example predictions [10, 12, 11, 13]
* Final prediction: (10 + 12 + 11 + 13) / 4 = 11.5

# Bagging and Pasting
Bagging and Pasting are two ensemble learning techniques used to train multiple models on different subsets of the training data.
## Bagging
Bagging means training multiple models on random samples of the dataset with replacement. Sampling is done with replacement, so the same data point can appear multiple times in a sample.

Original dataset:
[1,2,3,4,5]

Bagging samples might be:

* Sample 1 → [2,4,4,5,1]
* Sample 2 → [3,1,2,2,5]
* Sample 3 → [5,3,3,1,4]

## Pasting
Pasting is very similar to bagging, but sampling is done without replacement.

Original dataset:
[1,2,3,4,5]

Bagging samples might be:

* Sample 1 → [1,2,4]
* Sample 2 → [2,3,5]
* Sample 3 → [1,3,4]

# Random Forest

Random Forest is an ensemble learning algorithm that builds many decision trees and combines their predictions to produce a more accurate and stable result.Instead of relying on a single decision tree, Random Forest trains many trees and combines their predictions.

## How Random Forest Works

* Step 1: Bootstrap Sampling (Bagging)

From the original dataset, multiple bootstrap samples are created. If original dataset has 100 samples. Each tree is trained on a random sample of 100 points with replacement. Some samples may appear multiple times, and some may not appear.

* Step 2: Train Decision Trees

For each tree, a random subset of features is selected at every split. The tree grows independently. Example: If dataset has 10 features, the tree may only consider 3–4 features at each split. This randomness makes trees different from each other.

* Step 3: Repeat for Many Trees

Tree 1 Tree 2 Tree 3 Tree 4 Tree 5 ... Tree 100

Each tree learns different patterns from the data.

* Step 4: Combine Predictions

For Classification -> Use majority voting (hard voting).

For Regression -> Take the average of predictions.

https://www.youtube.com/watch?v=NxEHSAfFlK8&t=362s

look at the assembly ai video for random forest and decision tree before lecture

In [ ]:
"""
Decision Tree Implementation from Scratch
Supports both Classification and Regression
"""

import numpy as np
from collections import Counter


class Node:
    """
    Represents a node in the decision tree.
    Can be either a decision node (internal) or a leaf node.
    """
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature        # Index of feature to split on
        self.threshold = threshold    # Threshold value for split
        self.left = left             # Left child node
        self.right = right           # Right child node
        self.value = value           # Value if leaf node (prediction)

    def is_leaf_node(self):
        return self.value is not None


class DecisionTree:
    """
    Decision Tree for Classification and Regression

    Parameters:
    -----------
    min_samples_split : int, default=2
        Minimum number of samples required to split a node
    max_depth : int, default=100
        Maximum depth of the tree
    n_features : int, default=None
        Number of features to consider when looking for best split
        If None, then n_features = all features
    task : str, default='classification'
        Type of task: 'classification' or 'regression'
    """

    def __init__(self, min_samples_split=2, max_depth=100, n_features=None, task='classification'):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.n_features = n_features
        self.task = task
        self.root = None

    def fit(self, X, y):
        """
        Build decision tree from training data

        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Training data
        y : array-like, shape (n_samples,)
            Target values
        """
        # Determine number of features to use
        self.n_features = X.shape[1] if not self.n_features else min(X.shape[1], self.n_features)

        # Grow the tree
        self.root = self._grow_tree(X, y)

    def _grow_tree(self, X, y, depth=0):
        """
        Recursively grow the decision tree
        """











        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))

        # Check stopping criteria
        if (depth >= self.max_depth or
            n_labels == 1 or
            n_samples < self.min_samples_split):
            leaf_value = self._leaf_value(y)
            return Node(value=leaf_value)

        # Find best split
        feat_idxs = np.random.choice(n_feats, self.n_features, replace=False)


        best_feature, best_thresh = self._best_split(X, y, feat_idxs)

        # Check if valid split was found
        if best_feature is None:
            return Node(value=self._leaf_value(y))

        # Create child nodes
        left_idxs, right_idxs = self._split(X[:, best_feature], best_thresh)
        if len(left_idxs) == 0 or len(right_idxs) == 0:
          return Node(value=self._leaf_value(y))


        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)

        return Node(best_feature, best_thresh, left, right)

    def _best_split(self, X, y, feat_idxs):
        """
        Find the best feature and threshold to split on
        """
        best_gain = -1
        split_idx, split_threshold = None, None

        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)


            for thr in thresholds:
                # Calculate information gain
                gain = self._information_gain(y, X_column, thr)


                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_threshold = thr

        return split_idx, split_threshold

    def _information_gain(self, y, X_column, threshold):
        """
        Calculate information gain from a split
        """
        # Parent impurity
        parent_impurity = self._impurity(y)

        # Create children
        left_idxs, right_idxs = self._split(X_column, threshold)

        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0

        # Calculate weighted avg impurity of children
        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l, e_r = self._impurity(y[left_idxs]), self._impurity(y[right_idxs])
        child_impurity = (n_l / n) * e_l + (n_r / n) * e_r

        # Calculate information gain
        information_gain = parent_impurity - child_impurity
        return information_gain

    def _split(self, X_column, split_thresh):
        """
        Split data based on feature value and threshold
        """
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs

    def _impurity(self, y):
        """
        Calculate impurity measure based on task type
        """
        if self.task == 'classification':
            # Gini impurity for classification
            hist = np.bincount(y.astype(int))
            ps = hist / len(y)
            return 1 - np.sum(ps ** 2)
        else:
            # Variance for regression
            return np.var(y)

    def _leaf_value(self, y):
        """
        Calculate the value for a leaf node
        """
        if self.task == 'classification':
            # Most common class
            counter = Counter(y)
            value = counter.most_common(1)[0][0]
            return value
        else:
            # Mean value for regression
            return np.mean(y)

    def predict(self, X):
        """
        Predict class or value for X

        Parameters:
        -----------
        X : array-like, shape (n_samples, n_features)
            Samples to predict

        Returns:
        --------
        predictions : array, shape (n_samples,)
            Predicted values
        """
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        """
        Traverse tree to find prediction for a single sample
        """
        if node.is_leaf_node():
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)


# Convenience classes for specific tasks
class DecisionTreeClassifier(DecisionTree):
    def __init__(self, min_samples_split=2, max_depth=100, n_features=None):
        super().__init__(min_samples_split, max_depth, n_features, task='classification')


class DecisionTreeRegressor(DecisionTree):
    def __init__(self, min_samples_split=2, max_depth=100, n_features=None):
        super().__init__(min_samples_split, max_depth, n_features, task='regression')

In [ ]:
import numpy as np
from collections import Counter

class RandomForest:

    def __init__(self, n_trees=10, max_depth=100, min_samples_split=2, n_features=None, task='classification'):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features
        self.task = task
        self.trees = []

    def fit(self, X, y):

        self.trees = []

        for _ in range(self.n_trees):

            # Bootstrap sampling
            X_sample, y_sample = self._bootstrap_samples(X, y)

            tree = DecisionTree(
                min_samples_split=self.min_samples_split,
                max_depth=self.max_depth,
                n_features=self.n_features,
                task=self.task
            )

            tree.fit(X_sample, y_sample)

            self.trees.append(tree)

    def _bootstrap_samples(self, X, y):

        n_samples = X.shape[0]

        idxs = np.random.choice(n_samples, n_samples, replace=True)

        return X[idxs], y[idxs]

    def predict(self, X):

        # predictions from all trees
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        print("shape before swapping")
        print(tree_preds.shape)


            # Print each tree's predictions for the samples
        # for i, pred in enumerate(tree_preds):
        #   print(f"Tree {i+1} predictions: {pred}")

        #



        # transpose so shape becomes (n_samples, n_trees)
        tree_preds = np.swapaxes(tree_preds, 0, 1)
        print("shape after swapping")
        print(tree_preds.shape)

        if self.task == 'classification':
            return np.array([self._most_common_label(pred) for pred in tree_preds])

        else:
            return np.array([np.mean(pred) for pred in tree_preds])

    def _most_common_label(self, y):

        counter = Counter(y)

        return counter.most_common(1)[0][0]

In [ ]:
"""
Simple Manual Example: Understanding Decision Tree Logic
This demonstrates how a decision tree makes decisions step by step
"""

import numpy as np

print("="*60)
print("SIMPLE EXAMPLE: SHOULD WE PLAY TENNIS?")
print("="*60)


# High humidity: 85-96%
# Normal humidity: 65-80%
X_train = np.array([
    [0, 85, 85, 0],  # not Rainy, Hot, High humidity, Weak wind -> No
    [0, 80, 90, 1],  # not Rainy, Hot, High humidity, Strong wind -> No
    [0, 83, 78, 0],  # not Rainy, Hot, High humidity, Weak wind -> Yes
    [1, 70, 96, 0],  # Rainy, Mild, High humidity, Weak wind -> Yes
    [1, 68, 80, 0],  # Rainy, Cool, Normal humidity, Weak wind -> Yes
    [1, 65, 70, 1],  # Rainy, Cool, Normal humidity, Strong wind -> No
    [0, 64, 65, 1],  # not Rainy, Cool, Normal humidity, Strong wind -> Yes
    [0, 72, 95, 0],  # not Rainy, Mild, High humidity, Weak wind -> No
    [0, 69, 70, 0],  # not Rainy, Cool, Normal humidity, Weak wind -> Yes
    [1, 75, 80, 0],  # Rainy, Mild, Normal humidity, Weak wind -> Yes
    [0, 75, 70, 1],  # not Rainy, Mild, Normal humidity, Strong wind -> Yes
    [0, 72, 90, 1],  # not Rainy, Mild, High humidity, Strong wind -> Yes
    [0, 81, 75, 0],  # not Rainy, Hot, Normal humidity, Weak wind -> Yes
    [1, 71, 80, 1],  # Rainy, Mild, Normal humidity, Strong wind -> No
])

y_train = np.array([0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0])

feature_names = ['Outlook', 'Temperature', 'Humidity', 'Wind']

# Train the tree
# Train the Random Forest
rf = RandomForest(
    n_trees=10,
    max_depth=5,
    min_samples_split=2,
    n_features=2,
    task='classification'
)

rf.fit(X_train, y_train)

print("\nTraining completed!")
print(f"Total training samples: {len(y_train)}")
print(f"Play Tennis: {sum(y_train)} times")
print(f"Don't Play: {len(y_train) - sum(y_train)} times")


print("\n" + "="*60)
print("MAKING PREDICTIONS")
print("="*60)


test_cases = [
    ([0, 70, 80, 0], "not Rainy, Cool, High Humidity, Weak Wind"),
    ([0, 75, 75, 0], "not Rainy, Mild, Normal Humidity, Weak Wind"),
    ([1, 65, 90, 1], "Rainy, Cool, High Humidity, Strong Wind"),
]

X_test = np.array([features for features, _ in test_cases])
predictions = rf.predict(X_test)
for i, (_, description) in enumerate(test_cases):
    prediction = predictions[i]
    result = "Yes, Play Tennis! ✓" if prediction == 1 else "No, Don't Play ✗"

    print(f"\nWeather: {description}")
    print(f"Features: {X_test[i]}")
    print(f"Decision: {result}")


# for features, description in test_cases:
#     prediction = rf.predict(np.array([features]))[0]
#     result = "Yes, Play Tennis! ✓" if prediction == 1 else "No, Don't Play ✗"

#     print(f"\nWeather: {description}")
#     print(f"Features: {features}")
#     print(f"Decision: {result}")

SIMPLE EXAMPLE: SHOULD WE PLAY TENNIS?

Training completed!
Total training samples: 14
Play Tennis: 9 times
Don't Play: 5 times

MAKING PREDICTIONS
shape before swapping
(10, 3)
shape after swapping
(3, 10)

Weather: not Rainy, Cool, High Humidity, Weak Wind
Features: [ 0 70 80  0]
Decision: Yes, Play Tennis! ✓

Weather: not Rainy, Mild, Normal Humidity, Weak Wind
Features: [ 0 75 75  0]
Decision: Yes, Play Tennis! ✓

Weather: Rainy, Cool, High Humidity, Strong Wind
Features: [ 1 65 90  1]
Decision: No, Don't Play ✗
